In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

ROOT     = r"C:\Users\NIAMBELE Siata\Desktop\MSPR2"
RAW_ELEC = os.path.join(ROOT, "01_raw", "electoral")
RAW_SOCI = os.path.join(ROOT, "01_raw", "socioeco")

def lire_csv(path, nrows=None):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc, sep=None,
                             engine="python", on_bad_lines="skip", nrows=nrows)
            return df, enc
        except Exception:
            continue
    raise ValueError(f"Impossible de lire : {path}")

def lire_xlsx(path, sheet=0, skiprows=0):
    try:
        return pd.read_excel(path, sheet_name=sheet, skiprows=skiprows, engine="calamine")
    except Exception:
        return pd.read_excel(path, sheet_name=sheet, skiprows=skiprows, engine="openpyxl")

In [2]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

def lire_csv(path, nrows=None):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc, sep=None,
                             engine="python", on_bad_lines="skip", nrows=nrows)
            return df, enc
        except Exception:
            continue
    raise ValueError(f"Impossible de lire : {path}")

def lire_xlsx(path, sheet=0, skiprows=0):
    try:
        return pd.read_excel(path, sheet_name=sheet, skiprows=skiprows, engine="calamine")
    except Exception:
        return pd.read_excel(path, sheet_name=sheet, skiprows=skiprows, engine="openpyxl")

# ------------------------------------------------------------
# 1. Exploration des fichiers électoraux
# ------------------------------------------------------------
print("=== FICHIERS DANS RAW_ELEC ===")
for f in os.listdir(RAW_ELEC):
    print(" -", f)

# Fichier general_results
path_gr = os.path.join(RAW_ELEC, "general_results.csv")
df_gr, enc_gr = lire_csv(path_gr)
print("\n--- general_results ---")
print(f"lignes : {len(df_gr):,} | encodage : {enc_gr}")
print(f"colonnes : {list(df_gr.columns)[:10]}...")
print(f"elections : {sorted(df_gr['id_election'].unique())}")

# Fichier candidats_results
path_cand = os.path.join(RAW_ELEC, "candidats_results.csv")
df_cand, enc_cand = lire_csv(path_cand)
print("\n--- candidats_results ---")
print(f"lignes : {len(df_cand):,} | encodage : {enc_cand}")
print(f"colonnes : {list(df_cand.columns)[:10]}...")
print(f"nuances : {df_cand['nuance'].nunique()} | ex : {df_cand['nuance'].dropna().unique()[:5]}")

# Rhône (code 69)
if 'code_departement' in df_gr.columns:
    df_69 = df_gr[df_gr['code_departement'].astype(str).str.strip() == '69']
    print(f"\nRhône dans general_results : {len(df_69):,} lignes")
if 'code_departement' in df_cand.columns:
    df_cand_69 = df_cand[df_cand['code_departement'].astype(str).str.strip() == '69']
    print(f"Rhône dans candidats_results : {len(df_cand_69):,} lignes")

# ------------------------------------------------------------
# 2. Exploration des fichiers socio-économiques
# ------------------------------------------------------------
print("\n=== FICHIERS DANS RAW_SOCI ===")
for f in os.listdir(RAW_SOCI):
    print(" -", f)

=== FICHIERS DANS RAW_ELEC ===
 - candidats_results.csv
 - candidats_results_rhone69.csv
 - candidats_rhone_legi.csv
 - candidature_2022_T1.xlsx
 - candidature_2024_T1.csv
 - general_results.csv
 - general_results_rhone69.csv
 - legislatives-2024-candidatures-france-entiere-tour-1-2024-06-28.csv
 - README.md

--- general_results ---
lignes : 3,162,440 | encodage : utf-8
colonnes : ['id_election', 'id_brut_miom', 'code_departement', 'libelle_departement', 'code_canton', 'libelle_canton', 'code_commune', 'libelle_commune', 'code_circonscription', 'libelle_circonscription']...
elections : ['1999_euro_t1', '2001_cant_t1', '2001_cant_t2', '2002_legi_t1', '2002_legi_t2', '2002_pres_t1', '2002_pres_t2', '2004_cant_t1', '2004_cant_t2', '2004_euro_t1', '2004_regi_t1', '2004_regi_t2', '2007_legi_t1', '2007_legi_t2', '2007_pres_t1', '2007_pres_t2', '2008_cant_t1', '2008_cant_t2', '2008_muni_t1', '2008_muni_t2', '2009_euro_t1', '2010_regi_t1', '2010_regi_t2', '2011_cant_t1', '2011_cant_t2', '2012_

In [3]:
CANDIDATS = [
    ("candidature_2022_T1.xlsx", "xlsx", "2022", "T1"),
    ("candidature_2024_T1.csv",  "csv",  "2024", "T1"),
]

for nom, fmt, annee, tour in CANDIDATS:
    path = os.path.join(RAW_ELEC, nom)
    df = lire_csv(path)[0] if fmt == "csv" else lire_xlsx(path)
    col_dept = [c for c in df.columns if 'depart' in c.lower() and 'code' in c.lower()]
    n69 = (df[col_dept[0]].astype(str).str.strip().isin(['69','069'])).sum() if col_dept else "?"
    print(f"{nom} [{annee} {tour}]")
    print(f"  {len(df):,} lignes | {len(df.columns)} colonnes | Rhône: {n69}")
    print(f"  Cols: {list(df.columns)[:6]}...\n")

candidature_2022_T1.xlsx [2022 T1]
  6,290 lignes | 18 colonnes | Rhône: ?
  Cols: ['Code du département', 'Libellé du département', 'Code circonscription', 'Libellé circonscription', 'N° panneau', 'N° candidat']...

candidature_2024_T1.csv [2024 T1]
  4,009 lignes | 18 colonnes | Rhône: ?
  Cols: ['Code département', 'Département', 'Code circonscription', 'Libellé circonscription', 'Numéro de panneau', 'N° dépôt']...



In [5]:
print("=== Contenu du dossier socioeco ===\n")

# Lister tous les fichiers et sous-dossiers
if os.path.exists(RAW_SOCI):
    for item in os.listdir(RAW_SOCI):
        item_path = os.path.join(RAW_SOCI, item)
        if os.path.isdir(item_path):
            print(f"[DOSSIER] {item}")
        else:
            taille = os.path.getsize(item_path)
            print(f"[FICHIER] {item} ({taille} octets)")
    print(f"\nTotal : {len(os.listdir(RAW_SOCI))} éléments")
else:
    print(f"Le dossier {RAW_SOCI} n'existe pas.")

=== Contenu du dossier socioeco ===

[FICHIER] base-cc-emploi-pop-active-2020_v2.CSV (150327454 octets)
[FICHIER] BTX_TD_NAT1_2016.xls (10574336 octets)
[FICHIER] cc_filosofi_2017_COM.CSV (1842281 octets)
[FICHIER] delinquance_communale.csv (621164756 octets)
[FICHIER] FILO2021_DISP_COM.xlsx (92116466 octets)
[FICHIER] population.CSV (14824370 octets)
[FICHIER] README.md (283 octets)
[FICHIER] TD_IMG1A_2022.xlsx (6920567 octets)

Total : 8 éléments


BILAN DES FICHIERS

In [6]:
print(f"{'Zone':<12} {'Fichier':<55} {'Mo':>6}")
print("-" * 76)
for dossier, zone in [(RAW_ELEC, "electoral"), (RAW_SOCI, "socioeco")]:
    for f in sorted(os.listdir(dossier)):
        taille = os.path.getsize(os.path.join(dossier, f)) / (1024 * 1024)
        print(f"  {zone:<12} {f:<55} {taille:>5.1f}")

Zone         Fichier                                                     Mo
----------------------------------------------------------------------------
  electoral    README.md                                                 0.0
  electoral    candidats_results.csv                                   2273.3
  electoral    candidature_2022_T1.xlsx                                  0.7
  electoral    candidature_2024_T1.csv                                   0.6
  electoral    general_results.csv                                     386.9
  electoral    legislatives-2024-candidatures-france-entiere-tour-1-2024-06-28.csv   0.6
  socioeco     BTX_TD_NAT1_2016.xls                                     10.1
  socioeco     FILO2021_DISP_COM.xlsx                                   87.8
  socioeco     README.md                                                 0.0
  socioeco     TD_IMG1A_2022.xlsx                                        6.6
  socioeco     base-cc-emploi-pop-active-2020_v2.CSV            